# Ingesta de datos de agricultura en Sonora

Esta libreta de Jupyter se utilizará para la ingesta y análisis de datos relacionados con la agricultura en el estado de Sonora. Para esto utilizaremos dataset disponibles públicamente sobre cultivos, producción y condiciones climáticas en la región a lo largo de los años. Los datos obtenidos serán procesados y transformados para facilitar su análisis y visualización, permitiendo así obtener insights relevantes sobre la agricultura en Sonora.

## Ingesta de datos

### Librerias necesarias

In [14]:
!pip install pydantic


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from pathlib import Path
import pandas as pd
from loguru import logger
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pydantic import BaseModel, PositiveInt, ValidationError, Field, ConfigDict, field_validator
from typing import Optional
from tqdm import tqdm

RAW_DATA_DIR = Path("../data/raw")


SONORA_AGRICULTURE_URL = (
    "https://datos.sonora.gob.mx/dataset/3e17a7e8-c8d4-49c4-a426-4dec099cd0cd/"
    "resource/9e455a9c-1733-490e-ba3d-6117ffe4ffe5/download/"
    "agricultura-sonora-{year}.xlsx"
)

### Descarga de datos

In [16]:
def download_agriculture_data(start_year: int = 2019, end_year: int = 2024):
    """Descarga los datos de agricultura de Sonora por año."""
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    years = range(start_year, end_year + 1)

    for year in tqdm(years, total=len(years)):
        url = SONORA_AGRICULTURE_URL.format(year=year)
        dest_file = RAW_DATA_DIR / f"agricultura-sonora-{year}.xlsx"

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            with open(dest_file, "wb") as f:
                f.write(response.content)

            # logger.info(f"Downloaded agriculture {year} -> {dest_file}")
            logger.success(f"Año {year}: Descargado con éxito")

            print("Carpeta destino:", RAW_DATA_DIR)
            print("Existe:", RAW_DATA_DIR.exists())
        except requests.exceptions.RequestException as e:
            logger.warning(f"Failed to download agriculture {year}: {e}")

    logger.success("Agriculture download complete.")

In [17]:
download_agriculture_data(start_year=2019, end_year=2024)

 17%|█▋        | 1/6 [00:00<00:04,  1.16it/s]

Carpeta destino: ../data/raw
Existe: True


2026-09-26 17:26:40.908 | SUCCESS  | __main__:download_agriculture_data:19 - Año 2020: Descargado con éxito
 33%|███▎      | 2/6 [00:01<00:03,  1.22it/s]

Carpeta destino: ../data/raw
Existe: True


2026-09-26 17:26:41.474 | SUCCESS  | __main__:download_agriculture_data:19 - Año 2021: Descargado con éxito
 50%|█████     | 3/6 [00:02<00:02,  1.42it/s]

...

Carpeta destino: ../data/raw
Existe: True


2026-09-26 17:26:42.620 | SUCCESS  | __main__:download_agriculture_data:19 - Año 2023: Descargado con éxito
 83%|████████▎ | 5/6 [00:03<00:00,  1.60it/s]

Carpeta destino: ../data/raw
Existe: True


2026-09-26 17:26:43.193 | SUCCESS  | __main__:download_agriculture_data:19 - Año 2024: Descargado con éxito
100%|██████████| 6/6 [00:03<00:00,  1.52it/s]
2026-09-26 17:26:43.194 | SUCCESS  | __main__:download_agriculture_data:26 - Agriculture download complete.


Carpeta destino: ../data/raw
Existe: True


In [18]:
for year in range(2019, 2025):
    df = pd.read_excel(RAW_DATA_DIR / f"agricultura-sonora-{year}.xlsx")
    globals()[f"agricultura_sonora_df_{year}"] = df
    logger.info(f"Loaded agricultura-sonora-{year}.xlsx")

2026-09-26 17:26:43.281 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2019.xlsx
2026-09-26 17:26:43.350 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2020.xlsx
2026-09-26 17:26:43.418 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2021.xlsx
2026-09-26 17:26:43.482 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2022.xlsx
2026-09-26 17:26:43.543 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2023.xlsx
2026-09-26 17:26:43.604 | INFO     | __main__:<module>:4 - Loaded agricultura-sonora-2024.xlsx


In [19]:
agricultura_sonora_df_2019.head()

,ANO,CIERREYAVAN,CICLO,CDDR,NDDR,CMUN,NMUN,CVECUL,CULTIVO,SUPSEM,SUPCOSE,SUPSINI,PRODTON,RENDMNTO,PMR,VALPROD
0,2019,CIERRE DE PRODUCCION AGRICOLA 2019,3,140,DDR 140 MAGDALENA,64,Trincheras,901,Aceituna,2.0,2.0,0.0,8.2,4.1,13526.82,110.92
1,2019,CIERRE DE PRODUCCION AGRICOLA 2019,3,139,DDR 139 CABORCA,4,Altar,901,Aceituna,0.0,0.0,0.0,0.0,0.0,0.00,0.00
2,2019,CIERRE DE PRODUCCION AGRICOLA 2019,3,139,DDR 139 CABORCA,17,Caborca,901,Aceituna,1211.0,1211.0,0.0,11505.0,9.5,11600.00,133458.00
3,2019,CIERRE DE PRODUCCION AGRICOLA 2019,2,144,DDR 144 HERMOSILLO,30,Hermosillo,1,Acelga,2.0,2.0,0.0,20.0,10.0,10500.00,210.00
4,2019,CIERRE DE PRODUCCION AGRICOLA 2019,1,144,DDR 144 HERMOSILLO,56,San Miguel de Horcasitas,1,Acelga,8.0,8.0,0.0,148.0,18.5,9250.00,1369.00


## Validación de la descarga de datos de agricultura de Sonora

### Schema basandonos en el diccionario de datos de agricultura de Sonora

In [20]:
class AgricultureSchema(BaseModel):
    model_config = ConfigDict(extra="ignore")

    ANO: Optional[str] = None
    CIERREYAVAN: Optional[str] = None
    CICLO: Optional[int] = Field(default=None, ge=1, le=3)
    CDDR: Optional[str] = None
    NDDR: Optional[str] = None
    CMUN: Optional[str] = None
    NMUN: Optional[str] = None
    CVMES: Optional[str] = None
    NMES: Optional[str] = None
    CVECUL: Optional[str] = None
    CULTIVO: Optional[str] = None
    CVEVAR: Optional[str] = None
    DESVAR: Optional[str] = None
    SUPSEM: Optional[float] = Field(default=None, ge=0)
    SUPCOSE: Optional[float] = Field(default=None, ge=0)
    SUPSINI: Optional[float] = Field(default=None, ge=0)
    PRODTON: Optional[float] = Field(default=None, ge=0)
    RENDMNTO: Optional[float] = Field(default=None, ge=0)
    PMR: Optional[float] = Field(default=None, ge=0)
    VALPROD: Optional[float] = Field(default=None, ge=0)

    @field_validator("*")
    @classmethod
    def normalize_missing_values(cls, value):
        if value is None or pd.isna(value):
            return None
        return value

    @field_validator(
        "ANO", "CIERREYAVAN", "CDDR", "NDDR", "CMUN", "NMUN",
        "CVMES", "NMES", "CVECUL", "CULTIVO", "CVEVAR", "DESVAR",
        mode="before",
    )
    @classmethod
    def convert_text_fields(cls, value):
        if value is None or pd.isna(value):
            return None
        return str(value).strip()

### Función para validar los datos descargados de agricultura de Sonora

In [21]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = AgricultureSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [22]:
df_validated = validate_df(agricultura_sonora_df_2019)